# Notebook 04: Differentiable Rendering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase1/04_differentiable_rendering.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand why differentiable rendering is essential for 3DGS
2. Learn how gradients flow through the rendering pipeline
3. Implement a simple differentiable renderer in PyTorch
4. Optimize Gaussian parameters from image supervision
5. Visualize the optimization process

**Estimated Time**: 75 minutes

**Prerequisites**: Notebook 03 (Projection & Splatting)

---

## Setup

In [ ]:
import os
import sys

# Colab setup
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('3DGS-from-scratch'):
        !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    os.chdir('3DGS-from-scratch')
    !pip install -q plotly ipywidgets

# Path setup
for path in ['../../src', '../src', './src']:
    full_path = os.path.abspath(path)
    if os.path.exists(os.path.join(full_path, 'gaussian')):
        sys.path.insert(0, full_path)
        break

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from IPython.display import clear_output

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("Setup complete!")

## 1. Why Differentiable Rendering?

### The Core Idea of 3DGS

3D Gaussian Splatting uses **analysis-by-synthesis**:

```
3D Gaussians → Render → Predicted Image
                          ↓
                     Compare with
                          ↓
Ground Truth Image ← Loss ← Gradients → Update Gaussians
```

### Requirements for Optimization

To optimize Gaussian parameters, we need:
1. A **loss function** comparing rendered and ground truth images
2. **Gradients** of the loss with respect to Gaussian parameters
3. The entire rendering pipeline must be **differentiable**

### The Chain of Dependencies

$$\text{Gaussian Params} \xrightarrow{\text{Project}} \text{2D Splats} \xrightarrow{\text{Blend}} \text{Image} \xrightarrow{\text{Loss}} L$$

We need:
$$\frac{\partial L}{\partial \mu}, \frac{\partial L}{\partial s}, \frac{\partial L}{\partial q}, \frac{\partial L}{\partial \alpha}, \frac{\partial L}{\partial c}$$

## 2. Building a Differentiable Renderer

Let's build a minimal differentiable renderer step by step.

In [ ]:
class MinimalDifferentiableRenderer:
    """
    Minimal differentiable renderer for educational purposes.
    
    This renderer is intentionally simple to show the core concepts.
    """
    
    def __init__(self, height, width, background=None):
        self.height = height
        self.width = width
        self.background = background if background is not None else torch.ones(3)
        
        # Create pixel coordinate grid
        y = torch.arange(height, dtype=torch.float32)
        x = torch.arange(width, dtype=torch.float32)
        self.y_grid, self.x_grid = torch.meshgrid(y, x, indexing='ij')
    
    def to(self, device):
        self.background = self.background.to(device)
        self.x_grid = self.x_grid.to(device)
        self.y_grid = self.y_grid.to(device)
        return self
    
    def render_single_gaussian(
        self,
        mean: torch.Tensor,      # [2] image coordinates
        covariance: torch.Tensor, # [2, 2]
        opacity: torch.Tensor,    # scalar
        color: torch.Tensor,      # [3] RGB
    ) -> torch.Tensor:
        """
        Render a single 2D Gaussian to an image.
        
        Returns:
            Image [H, W, 3]
        """
        # Compute inverse covariance
        cov_inv = torch.linalg.inv(covariance)
        
        # Offset from mean
        dx = self.x_grid - mean[0]
        dy = self.y_grid - mean[1]
        
        # Mahalanobis distance
        mahal = (
            cov_inv[0, 0] * dx * dx +
            (cov_inv[0, 1] + cov_inv[1, 0]) * dx * dy +
            cov_inv[1, 1] * dy * dy
        )
        
        # Gaussian value
        gaussian_value = torch.exp(-0.5 * mahal)
        
        # Apply opacity
        alpha = gaussian_value * opacity
        
        # Blend with background
        image = torch.zeros(self.height, self.width, 3, device=mean.device)
        for c in range(3):
            image[:, :, c] = alpha * color[c] + (1 - alpha) * self.background[c]
        
        return image


# Test the renderer
renderer = MinimalDifferentiableRenderer(100, 150)

# Create a Gaussian with requires_grad=True
mean = torch.tensor([75., 50.], requires_grad=True)
cov = torch.tensor([[400., 0.], [0., 200.]], requires_grad=True)
opacity = torch.tensor(0.9, requires_grad=True)
color = torch.tensor([1., 0., 0.], requires_grad=True)  # Red

# Render
image = renderer.render_single_gaussian(mean, cov, opacity, color)

print(f"Output shape: {image.shape}")
print(f"Mean has grad_fn: {mean.grad_fn is not None}")
print(f"Image has grad_fn: {image.grad_fn is not None}")

# Visualize
plt.figure(figsize=(8, 5))
plt.imshow(image.detach().numpy())
plt.scatter(mean[0].item(), mean[1].item(), c='white', s=100, marker='+')
plt.title('Rendered Gaussian (Differentiable)')
plt.colorbar(label='Intensity')
plt.show()

## 3. Computing Gradients

Let's verify that gradients flow through the renderer.

In [ ]:
# Create a target image (green circle in different position)
target_mean = torch.tensor([100., 60.])
target_cov = torch.tensor([[300., 0.], [0., 300.]])
target_opacity = torch.tensor(0.95)
target_color = torch.tensor([0., 1., 0.])  # Green

with torch.no_grad():
    target_image = renderer.render_single_gaussian(
        target_mean, target_cov, target_opacity, target_color
    )

# Initialize parameters (different from target)
mean = torch.tensor([50., 30.], requires_grad=True)
cov = torch.tensor([[200., 0.], [0., 200.]], requires_grad=True)
opacity = torch.tensor(0.8, requires_grad=True)
color = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)

# Render current state
pred_image = renderer.render_single_gaussian(mean, cov, opacity, color)

# Compute loss (MSE)
loss = F.mse_loss(pred_image, target_image)

# Backward pass
loss.backward()

print("Gradients after backward pass:")
print("=" * 50)
print(f"d(loss)/d(mean) = {mean.grad}")
print(f"d(loss)/d(cov) = \n{cov.grad}")
print(f"d(loss)/d(opacity) = {opacity.grad}")
print(f"d(loss)/d(color) = {color.grad}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(target_image.numpy())
axes[0].set_title('Target Image')
axes[0].scatter(target_mean[0], target_mean[1], c='white', s=100, marker='+')

axes[1].imshow(pred_image.detach().numpy())
axes[1].set_title('Predicted Image (Initial)')
axes[1].scatter(mean[0].item(), mean[1].item(), c='white', s=100, marker='+')

diff = torch.abs(pred_image - target_image).mean(dim=-1)
axes[2].imshow(diff.detach().numpy(), cmap='hot')
axes[2].set_title(f'Difference (Loss: {loss.item():.4f})')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print(f"- Mean gradient ({mean.grad.tolist()}) points toward target position")
print(f"- Color gradient ({color.grad.tolist()}) pushes toward target color (green)")

## 4. Optimization Demo: Fitting a Single Gaussian

Let's optimize a Gaussian to match a target image.

In [ ]:
def optimize_gaussian(
    renderer,
    target_image,
    init_mean,
    init_cov,
    init_opacity,
    init_color,
    n_iterations=200,
    lr=1.0,
    visualize_every=20,
):
    """
    Optimize Gaussian parameters to match target image.
    """
    # Initialize parameters
    mean = init_mean.clone().requires_grad_(True)
    cov = init_cov.clone().requires_grad_(True)
    opacity = init_opacity.clone().requires_grad_(True)
    color = init_color.clone().requires_grad_(True)
    
    # Optimizer
    optimizer = torch.optim.Adam([
        {'params': mean, 'lr': lr * 0.5},
        {'params': cov, 'lr': lr * 0.1},
        {'params': opacity, 'lr': lr * 0.05},
        {'params': color, 'lr': lr * 0.1},
    ])
    
    losses = []
    history = []
    
    for i in range(n_iterations):
        optimizer.zero_grad()
        
        # Ensure covariance is positive definite
        cov_safe = cov @ cov.T + torch.eye(2) * 10
        
        # Clamp opacity
        opacity_safe = torch.clamp(opacity, 0.01, 0.99)
        
        # Clamp color
        color_safe = torch.clamp(color, 0, 1)
        
        # Render
        pred_image = renderer.render_single_gaussian(
            mean, cov_safe, opacity_safe, color_safe
        )
        
        # Loss
        loss = F.mse_loss(pred_image, target_image)
        
        # Backward
        loss.backward()
        
        # Update
        optimizer.step()
        
        losses.append(loss.item())
        
        if i % visualize_every == 0 or i == n_iterations - 1:
            history.append({
                'iter': i,
                'loss': loss.item(),
                'image': pred_image.detach().clone(),
                'mean': mean.detach().clone(),
                'color': color_safe.detach().clone(),
            })
    
    return losses, history


# Create target
target_mean = torch.tensor([100., 50.])
target_cov = torch.tensor([[500., 100.], [100., 300.]])
target_opacity = torch.tensor(0.9)
target_color = torch.tensor([0.2, 0.7, 0.3])  # Greenish

with torch.no_grad():
    target_image = renderer.render_single_gaussian(
        target_mean, target_cov, target_opacity, target_color
    )

# Initialize far from target
init_mean = torch.tensor([30., 70.])
init_cov = torch.tensor([[10., 0.], [0., 10.]])
init_opacity = torch.tensor(0.5)
init_color = torch.tensor([0.8, 0.2, 0.2])  # Reddish

# Optimize
losses, history = optimize_gaussian(
    renderer, target_image,
    init_mean, init_cov, init_opacity, init_color,
    n_iterations=300, lr=2.0, visualize_every=30
)

In [ ]:
# Visualize optimization
fig = plt.figure(figsize=(16, 8))

# Loss curve
ax1 = fig.add_subplot(2, 4, 1)
ax1.semilogy(losses)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Loss (log scale)')
ax1.set_title('Optimization Progress')
ax1.grid(True, alpha=0.3)

# Target
ax2 = fig.add_subplot(2, 4, 2)
ax2.imshow(target_image.numpy())
ax2.set_title('Target')
ax2.axis('off')

# Evolution snapshots
for idx, h in enumerate(history[:6]):
    ax = fig.add_subplot(2, 4, idx + 3)
    ax.imshow(h['image'].numpy())
    ax.set_title(f"Iter {h['iter']}, Loss: {h['loss']:.4f}")
    ax.axis('off')

plt.tight_layout()
plt.show()

# Compare initial and final
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(history[0]['image'].numpy())
axes[0].set_title(f"Initial (Loss: {history[0]['loss']:.4f})")

axes[1].imshow(history[-1]['image'].numpy())
axes[1].set_title(f"Final (Loss: {history[-1]['loss']:.4f})")

axes[2].imshow(target_image.numpy())
axes[2].set_title('Target')

diff = torch.abs(history[-1]['image'] - target_image).mean(dim=-1)
axes[3].imshow(diff.numpy(), cmap='hot')
axes[3].set_title('Final - Target Difference')

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Multiple Gaussians: Alpha Blending

Real 3DGS uses many Gaussians with proper alpha blending.

In [ ]:
class DifferentiableMultiGaussianRenderer:
    """
    Differentiable renderer for multiple Gaussians with alpha blending.
    """
    
    def __init__(self, height, width, background=None):
        self.height = height
        self.width = width
        self.background = background if background is not None else torch.ones(3)
        
        y = torch.arange(height, dtype=torch.float32)
        x = torch.arange(width, dtype=torch.float32)
        self.y_grid, self.x_grid = torch.meshgrid(y, x, indexing='ij')
    
    def to(self, device):
        self.background = self.background.to(device)
        self.x_grid = self.x_grid.to(device)
        self.y_grid = self.y_grid.to(device)
        return self
    
    def render(
        self,
        means: torch.Tensor,      # [N, 2]
        covariances: torch.Tensor, # [N, 2, 2]
        opacities: torch.Tensor,   # [N]
        colors: torch.Tensor,      # [N, 3]
        depths: torch.Tensor,      # [N] for sorting
    ) -> torch.Tensor:
        """
        Render multiple Gaussians with proper alpha blending.
        
        Uses front-to-back blending:
        C = Σ (T_i * α_i * c_i) + T_final * background
        
        where T_i = Π_{j<i} (1 - α_j)
        """
        N = means.shape[0]
        device = means.device
        
        # Sort by depth (front to back)
        depth_order = torch.argsort(depths)
        means = means[depth_order]
        covariances = covariances[depth_order]
        opacities = opacities[depth_order]
        colors = colors[depth_order]
        
        # Initialize
        image = torch.zeros(self.height, self.width, 3, device=device)
        transmittance = torch.ones(self.height, self.width, device=device)
        
        for i in range(N):
            mean = means[i]
            cov = covariances[i]
            opacity = opacities[i]
            color = colors[i]
            
            # Compute inverse covariance
            cov_inv = torch.linalg.inv(cov)
            
            # Offset from mean
            dx = self.x_grid - mean[0]
            dy = self.y_grid - mean[1]
            
            # Mahalanobis distance
            mahal = (
                cov_inv[0, 0] * dx * dx +
                (cov_inv[0, 1] + cov_inv[1, 0]) * dx * dy +
                cov_inv[1, 1] * dy * dy
            )
            
            # Gaussian value and alpha
            gaussian_value = torch.exp(-0.5 * mahal)
            alpha = gaussian_value * opacity
            
            # Weighted contribution: T * α
            weight = transmittance * alpha
            
            # Accumulate color
            for c in range(3):
                image[:, :, c] = image[:, :, c] + weight * color[c]
            
            # Update transmittance: T = T * (1 - α)
            transmittance = transmittance * (1 - alpha)
        
        # Add background
        for c in range(3):
            image[:, :, c] = image[:, :, c] + transmittance * self.background[c]
        
        return torch.clamp(image, 0, 1)


# Test with multiple Gaussians
multi_renderer = DifferentiableMultiGaussianRenderer(100, 150)

# Create some overlapping Gaussians
N = 5
means = torch.tensor([
    [40., 50.],
    [75., 40.],
    [75., 60.],
    [110., 50.],
    [75., 50.],
], requires_grad=True)

covariances = torch.stack([
    torch.tensor([[400., 0.], [0., 300.]]),
    torch.tensor([[200., 50.], [50., 200.]]),
    torch.tensor([[300., -50.], [-50., 200.]]),
    torch.tensor([[250., 0.], [0., 400.]]),
    torch.tensor([[150., 0.], [0., 150.]]),
]).requires_grad_(True)

opacities = torch.tensor([0.7, 0.6, 0.6, 0.7, 0.9], requires_grad=True)

colors = torch.tensor([
    [1., 0., 0.],  # Red
    [0., 1., 0.],  # Green
    [0., 0., 1.],  # Blue
    [1., 1., 0.],  # Yellow
    [1., 1., 1.],  # White
], requires_grad=True)

depths = torch.tensor([3., 2., 2., 3., 1.])  # White is closest

# Render
image = multi_renderer.render(means, covariances, opacities, colors, depths)

# Visualize
plt.figure(figsize=(10, 6))
plt.imshow(image.detach().numpy())
plt.scatter(means[:, 0].detach(), means[:, 1].detach(), 
           c=colors.detach().numpy(), s=100, edgecolors='black')
plt.title('Multiple Gaussians with Alpha Blending')
plt.colorbar(label='Intensity')
plt.show()

print(f"\nRendered image shape: {image.shape}")
print(f"Has gradient: {image.grad_fn is not None}")

In [ ]:
# Verify gradients flow through multiple Gaussians
loss = image.sum()  # Simple loss for gradient checking
loss.backward()

print("Gradients for multiple Gaussians:")
print("=" * 50)
print(f"d(loss)/d(means) shape: {means.grad.shape}")
print(f"d(loss)/d(means):\n{means.grad}")
print(f"\nd(loss)/d(opacities): {opacities.grad}")
print(f"\nd(loss)/d(colors):\n{colors.grad}")

## 6. Optimizing Multiple Gaussians

Let's optimize multiple Gaussians to reconstruct a target image.

In [ ]:
# Create a more complex target
def create_target_image(height, width):
    """Create a target image with several colored shapes."""
    image = torch.ones(height, width, 3)  # White background
    
    y_grid, x_grid = torch.meshgrid(
        torch.arange(height, dtype=torch.float32),
        torch.arange(width, dtype=torch.float32),
        indexing='ij'
    )
    
    # Red circle
    mask1 = ((x_grid - 40)**2 + (y_grid - 50)**2) < 400
    image[mask1] = torch.tensor([0.9, 0.1, 0.1])
    
    # Green circle
    mask2 = ((x_grid - 110)**2 + (y_grid - 50)**2) < 500
    image[mask2] = torch.tensor([0.1, 0.8, 0.2])
    
    # Blue ellipse
    mask3 = ((x_grid - 75)**2 / 600 + (y_grid - 50)**2 / 200) < 1
    image[mask3] = torch.tensor([0.2, 0.3, 0.9])
    
    return image

target = create_target_image(100, 150)

plt.figure(figsize=(8, 5))
plt.imshow(target.numpy())
plt.title('Target Image')
plt.axis('off')
plt.show()

In [ ]:
class GaussianOptimizer:
    """
    Optimizer for multiple Gaussians.
    """
    
    def __init__(self, n_gaussians, height, width, device='cpu'):
        self.n_gaussians = n_gaussians
        self.renderer = DifferentiableMultiGaussianRenderer(height, width)
        self.renderer.to(device)
        self.device = device
        
        # Initialize parameters
        self.means = nn.Parameter(
            torch.rand(n_gaussians, 2) * torch.tensor([width, height])
        )
        
        # Use Cholesky factor for covariance (ensures positive definite)
        self.L = nn.Parameter(torch.randn(n_gaussians, 2, 2) * 5)
        
        # Raw parameters for activation
        self.opacities_raw = nn.Parameter(torch.zeros(n_gaussians))
        self.colors_raw = nn.Parameter(torch.randn(n_gaussians, 3) * 0.1)
        
        # Depths (fixed for 2D case)
        self.depths = torch.linspace(0, 1, n_gaussians)
    
    @property
    def covariances(self):
        """Compute covariances from Cholesky factors."""
        # Create modified L without in-place operations
        L = self.L.clone()
        L_diag_0 = torch.abs(self.L[:, 0, 0]) + 5
        L_diag_1 = torch.abs(self.L[:, 1, 1]) + 5
        
        # Build L matrix properly
        L_safe = torch.zeros_like(self.L)
        L_safe[:, 0, 0] = L_diag_0
        L_safe[:, 1, 1] = L_diag_1
        L_safe[:, 0, 1] = 0  # Lower triangular
        
        return L_safe @ L_safe.transpose(-1, -2)
    
    @property
    def opacities(self):
        return torch.sigmoid(self.opacities_raw)
    
    @property
    def colors(self):
        return torch.sigmoid(self.colors_raw)
    
    def render(self):
        return self.renderer.render(
            self.means, self.covariances, self.opacities, self.colors, self.depths
        )
    
    def parameters(self):
        return [self.means, self.L, self.opacities_raw, self.colors_raw]


# Create optimizer with 20 Gaussians
N_GAUSSIANS = 20
gaussian_opt = GaussianOptimizer(N_GAUSSIANS, 100, 150)

# Setup optimizer
optimizer = torch.optim.Adam(gaussian_opt.parameters(), lr=1.0)

# Training loop
n_iterations = 500
losses = []
history = []

for i in range(n_iterations):
    optimizer.zero_grad()
    
    # Render
    pred = gaussian_opt.render()
    
    # Loss
    loss = F.mse_loss(pred, target)
    
    # Backward and update
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if i % 50 == 0 or i == n_iterations - 1:
        history.append({
            'iter': i,
            'loss': loss.item(),
            'image': pred.detach().clone(),
            'means': gaussian_opt.means.detach().clone(),
            'colors': gaussian_opt.colors.detach().clone(),
        })
        print(f"Iter {i:4d}: Loss = {loss.item():.6f}")

In [ ]:
# Visualize results
fig = plt.figure(figsize=(16, 10))

# Loss curve
ax1 = fig.add_subplot(2, 3, 1)
ax1.semilogy(losses)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Loss (log scale)')
ax1.set_title('Training Progress')
ax1.grid(True, alpha=0.3)

# Target
ax2 = fig.add_subplot(2, 3, 2)
ax2.imshow(target.numpy())
ax2.set_title('Target')
ax2.axis('off')

# Evolution
for idx, h in enumerate(history[:4]):
    ax = fig.add_subplot(2, 3, idx + 3)
    ax.imshow(h['image'].numpy())
    ax.scatter(h['means'][:, 0].numpy(), h['means'][:, 1].numpy(),
              c=h['colors'].numpy(), s=30, edgecolors='black', alpha=0.7)
    ax.set_title(f"Iter {h['iter']}, Loss: {h['loss']:.6f}")
    ax.axis('off')

plt.tight_layout()
plt.show()

# Final comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(target.numpy())
axes[0].set_title('Target')

final = history[-1]['image']
axes[1].imshow(final.numpy())
axes[1].scatter(history[-1]['means'][:, 0].numpy(), 
               history[-1]['means'][:, 1].numpy(),
               c=history[-1]['colors'].numpy(), s=50, 
               edgecolors='black', linewidths=1)
axes[1].set_title(f"Final ({N_GAUSSIANS} Gaussians)")

diff = torch.abs(final - target).mean(dim=-1)
axes[2].imshow(diff.numpy(), cmap='hot')
axes[2].set_title(f'Difference (MSE: {losses[-1]:.6f})')

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 7. The Gradient Flow in 3DGS

Let's visualize how gradients flow through the rendering pipeline.

In [ ]:
def visualize_gradient_flow():
    """
    Visualize the computational graph and gradient flow.
    """
    print("3DGS Gradient Flow:")
    print("=" * 60)
    print("""
    ┌─────────────────────────────────────────────────────────┐
    │                    FORWARD PASS                         │
    └─────────────────────────────────────────────────────────┘
    
    Gaussian Parameters (learnable)
    ├── μ (mean)           [N, 3]
    ├── s (scale)          [N, 3]     ──┐
    ├── q (rotation)       [N, 4]     ──┼──► Σ = R(q) · diag(s²) · R(q)ᵀ
    ├── α (opacity)        [N]        ──┘
    └── c (color/SH)       [N, 3/48]
                  │
                  ▼
         ┌────────────────┐
         │   Projection   │  μ_cam = R·μ + t
         │                │  μ_2d = project(μ_cam)
         │   Σ_2d = J·Σ·Jᵀ│
         └────────────────┘
                  │
                  ▼
         ┌────────────────┐
         │  Rasterization │  Evaluate Gaussian at pixels
         │                │  G(x) = exp(-½ d(x))
         └────────────────┘
                  │
                  ▼
         ┌────────────────┐
         │ Alpha Blending │  C = Σ Tᵢ·αᵢ·cᵢ + T_final·bg
         │                │  Tᵢ = Π_{j<i}(1-αⱼ)
         └────────────────┘
                  │
                  ▼
           Rendered Image
                  │
                  ▼
         ┌────────────────┐
         │     Loss       │  L = MSE(pred, target) + λ·DSSIM
         └────────────────┘
                  │
    
    ┌─────────────────────────────────────────────────────────┐
    │                   BACKWARD PASS                         │
    └─────────────────────────────────────────────────────────┘
    
                  │ ∂L/∂I (gradient of loss w.r.t. image)
                  ▼
         ┌────────────────┐
         │ Alpha Blending │  ∂L/∂α, ∂L/∂c
         └────────────────┘
                  │
                  ▼
         ┌────────────────┐
         │  Rasterization │  ∂L/∂μ_2d, ∂L/∂Σ_2d
         └────────────────┘
                  │
                  ▼
         ┌────────────────┐
         │   Projection   │  ∂L/∂μ, ∂L/∂Σ (chain rule through J)
         └────────────────┘
                  │
                  ▼
    Gaussian Parameter Gradients
    ├── ∂L/∂μ  ───► Update mean position
    ├── ∂L/∂s  ───► Update scale
    ├── ∂L/∂q  ───► Update rotation
    ├── ∂L/∂α  ───► Update opacity
    └── ∂L/∂c  ───► Update color/SH
    """)

visualize_gradient_flow()

In [ ]:
# Demonstrate gradient accumulation for a single Gaussian
print("\nGradient Accumulation Example:")
print("=" * 60)

# Simple 2D case
mean = torch.tensor([75., 50.], requires_grad=True)
cov = torch.tensor([[400., 0.], [0., 200.]], requires_grad=True)
opacity = torch.tensor(0.8, requires_grad=True)
color = torch.tensor([1., 0.5, 0.], requires_grad=True)

# Target: we want the Gaussian at a different position
target_pixel_value = torch.tensor([0., 0., 1.])  # Blue
pixel_location = torch.tensor([100., 50.])

# Compute Gaussian value at target pixel
cov_inv = torch.linalg.inv(cov)
diff = pixel_location - mean
mahal = diff @ cov_inv @ diff
gaussian_value = torch.exp(-0.5 * mahal)

# Alpha and blended color at that pixel
alpha = gaussian_value * opacity
pixel_color = alpha * color + (1 - alpha) * torch.ones(3)

# Loss: make this pixel blue
loss = F.mse_loss(pixel_color, target_pixel_value)

# Backward
loss.backward()

print(f"Target: Make pixel at {pixel_location.tolist()} blue")
print(f"\nCurrent values:")
print(f"  Gaussian mean: {mean.tolist()}")
print(f"  Gaussian value at target: {gaussian_value.item():.4f}")
print(f"  Pixel color: {pixel_color.detach().tolist()}")
print(f"  Loss: {loss.item():.4f}")

print(f"\nGradients:")
print(f"  ∂L/∂mean = {mean.grad.tolist()}")
print(f"  ∂L/∂color = {color.grad.tolist()}")
print(f"  ∂L/∂opacity = {opacity.grad.item():.4f}")

print("\nInterpretation:")
print(f"  - Mean gradient points toward target pixel")
print(f"  - Color gradient pushes R/G down, B up (toward blue)")
print(f"  - Opacity gradient: {'increase' if opacity.grad < 0 else 'decrease'} opacity")

## 8. Loss Functions in 3DGS

3DGS uses a combination of losses for optimization.

In [ ]:
def gaussian_kernel(window_size, sigma):
    """Create a Gaussian kernel for SSIM."""
    x = torch.arange(window_size, dtype=torch.float32) - window_size // 2
    gauss = torch.exp(-x**2 / (2 * sigma**2))
    kernel = gauss / gauss.sum()
    return kernel


def ssim(
    img1: torch.Tensor,
    img2: torch.Tensor,
    window_size: int = 11,
    sigma: float = 1.5,
) -> torch.Tensor:
    """
    Compute Structural Similarity Index (SSIM).
    
    SSIM compares images based on:
    - Luminance (mean)
    - Contrast (variance)
    - Structure (covariance)
    
    Args:
        img1, img2: Images [H, W, C] in range [0, 1]
        window_size: Size of Gaussian window
        sigma: Standard deviation of Gaussian window
    
    Returns:
        SSIM value (higher is better, max=1)
    """
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    
    # Create Gaussian window
    kernel_1d = gaussian_kernel(window_size, sigma)
    kernel_2d = kernel_1d.unsqueeze(1) * kernel_1d.unsqueeze(0)
    
    # Compute means using convolution
    def conv2d(img, kernel):
        # Simple 2D convolution (padding same)
        pad = window_size // 2
        result = torch.zeros_like(img)
        img_padded = F.pad(img.permute(2, 0, 1).unsqueeze(0), 
                          (pad, pad, pad, pad), mode='reflect')
        
        for c in range(img.shape[2]):
            result[:, :, c] = F.conv2d(
                img_padded[:, c:c+1], kernel.unsqueeze(0).unsqueeze(0)
            ).squeeze()
        return result
    
    mu1 = conv2d(img1, kernel_2d)
    mu2 = conv2d(img2, kernel_2d)
    
    mu1_sq = mu1 ** 2
    mu2_sq = mu2 ** 2
    mu1_mu2 = mu1 * mu2
    
    sigma1_sq = conv2d(img1 ** 2, kernel_2d) - mu1_sq
    sigma2_sq = conv2d(img2 ** 2, kernel_2d) - mu2_sq
    sigma12 = conv2d(img1 * img2, kernel_2d) - mu1_mu2
    
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    
    return ssim_map.mean()


def d_ssim(img1, img2):
    """Compute D-SSIM loss (1 - SSIM) / 2."""
    return (1 - ssim(img1, img2)) / 2


def combined_loss(pred, target, lambda_dssim=0.2):
    """
    3DGS loss function: L1 + λ·D-SSIM
    
    Args:
        pred: Predicted image [H, W, 3]
        target: Target image [H, W, 3]
        lambda_dssim: Weight for D-SSIM term
    
    Returns:
        Combined loss value
    """
    l1_loss = F.l1_loss(pred, target)
    dssim_loss = d_ssim(pred, target)
    return (1 - lambda_dssim) * l1_loss + lambda_dssim * dssim_loss


# Test losses
print("Loss Function Comparison:")
print("=" * 60)

# Create test images
target_test = target.clone()
pred_good = target_test + torch.randn_like(target_test) * 0.05  # Small noise
pred_bad = target_test + torch.randn_like(target_test) * 0.3   # Large noise
pred_shift = torch.roll(target_test, shifts=10, dims=1)         # Shifted

pred_good = torch.clamp(pred_good, 0, 1)
pred_bad = torch.clamp(pred_bad, 0, 1)

print(f"\n{'Prediction':<20} {'MSE':<12} {'L1':<12} {'SSIM':<12} {'Combined':<12}")
print("-" * 68)

for name, pred in [('Good (low noise)', pred_good), 
                   ('Bad (high noise)', pred_bad),
                   ('Shifted', pred_shift)]:
    mse = F.mse_loss(pred, target_test).item()
    l1 = F.l1_loss(pred, target_test).item()
    ssim_val = ssim(pred, target_test).item()
    comb = combined_loss(pred, target_test).item()
    print(f"{name:<20} {mse:<12.4f} {l1:<12.4f} {ssim_val:<12.4f} {comb:<12.4f}")

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

axes[0, 0].imshow(target_test.numpy())
axes[0, 0].set_title('Target')

for idx, (name, pred) in enumerate([('Good', pred_good), ('Bad', pred_bad), ('Shifted', pred_shift)]):
    axes[0, idx+1].imshow(pred.numpy())
    axes[0, idx+1].set_title(f'{name}')
    
    diff = torch.abs(pred - target_test).mean(dim=-1)
    axes[1, idx+1].imshow(diff.numpy(), cmap='hot', vmin=0, vmax=0.5)
    axes[1, idx+1].set_title(f'Difference')

axes[1, 0].axis('off')

for ax in axes.flat:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 9. Summary: Differentiable Rendering in 3DGS

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Differentiability** | All operations support gradient computation |
| **Analysis-by-synthesis** | Render → Compare → Update parameters |
| **Alpha blending** | Front-to-back compositing with transmittance |
| **Loss functions** | Combination of L1/MSE and perceptual (SSIM) losses |

### The Gradient Chain

$$\frac{\partial L}{\partial \theta} = \frac{\partial L}{\partial I} \cdot \frac{\partial I}{\partial \text{blend}} \cdot \frac{\partial \text{blend}}{\partial G_{2D}} \cdot \frac{\partial G_{2D}}{\partial \text{proj}} \cdot \frac{\partial \text{proj}}{\partial \theta}$$

where $\theta = \{\mu, s, q, \alpha, c\}$ are the Gaussian parameters.

### Why It Works

1. **PyTorch autograd** handles the chain rule automatically
2. **Smooth operations** (Gaussian, projection) have well-defined gradients
3. **Alpha blending** is differentiable with respect to all inputs

---

## Key Takeaways

1. Differentiable rendering enables optimization from image supervision
2. All components (projection, splatting, blending) must be differentiable
3. PyTorch autograd provides automatic gradient computation
4. Combined L1 + D-SSIM loss balances pixel accuracy and perceptual quality

---

## Next Steps

In the next notebook, we'll dive deeper into **alpha blending** and the rendering equation:

**[05_alpha_blending.ipynb](./05_alpha_blending.ipynb)** - Alpha Blending and Volume Rendering